<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/02-rdd-fundamentals/03-partitions-repartition-coalesce.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 2.3 — Partitions, repartition() vs coalesce()

`glom()` is the star of this notebook: it shows you exactly where every
element lives. Spark UI open, as always.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("2.3-partitions")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

## Partitions, visualized with glom()

In [ ]:
rdd = sc.parallelize(range(12), 4)
rdd.glom().collect()   # 4 lists = 4 partitions = 4 tasks per stage

In [ ]:
# repartition: full shuffle, even results, works both directions
print("up:  ", rdd.repartition(6).glom().collect())
print("down:", rdd.repartition(2).glom().collect())

In [ ]:
# coalesce: merges in place — and silently refuses to increase
print("coalesce(2): ", rdd.coalesce(2).glom().collect())
print("coalesce(16):", rdd.coalesce(16).getNumPartitions(), "partitions — still 4!")

## coalesce doesn't rebalance — it can make unevenness worse

In [ ]:
# Build deliberately uneven partitions: sizes 1, 2, 3, 94
uneven = sc.parallelize(
    [0]*1 + [1]*2 + [2]*3 + [3]*94, 4
)
sizes = lambda r: [len(p) for p in r.glom().collect()]
print("original:        ", sizes(uneven))
print("coalesce(2):     ", sizes(uneven.coalesce(2)),    "<- glued as-is")
print("repartition(2):  ", sizes(uneven.repartition(2)), "<- rebalanced (paid a shuffle)")

## The parallelism experiment

Same computation, three partition counts. Watch task counts and durations
in the Stages tab. (`f` burns CPU so the difference is visible.)

In [ ]:
import time

def slow_square(x):
    for _ in range(300):   # artificial CPU work
        x2 = (x * x) % 999983
    return x2

data = list(range(200_000))
for n in (2, 8, 64):
    t0 = time.perf_counter()
    sc.parallelize(data, n).map(slow_square).count()
    print(f"{n:>3} partitions: {time.perf_counter() - t0:.2f}s")

Expect: 2 is slowest (cores idle), 8 ≈ best on an 8-core machine, 64 fine
or slightly worse (scheduling overhead). Your numbers depend on your CPU.

## The post-filter crumbs pattern

In [ ]:
big      = sc.parallelize(range(1_000_000), 64)
filtered = big.filter(lambda x: x % 1000 == 0)     # keeps 0.1%
print("after filter:", filtered.getNumPartitions(), "partitions for", filtered.count(), "elements")

fixed = filtered.coalesce(4)
print("after coalesce:", fixed.getNumPartitions(), "partitions")

## Exercises

1. Predict `sizes()` for `uneven.coalesce(3)` before running it. Which partitions got glued?
2. In the UI, compare the job from `repartition(2).count()` vs `coalesce(2).count()` on `big` — which one has an extra stage, and why?
3. `sc.parallelize(range(100), 8).partitionBy(4)` fails — why? Fix it by making the RDD a pair RDD first, then use `glom()` to verify all equal keys share a partition.

In [ ]:
# your exercise code here
